### Import libraries
os, gc, random, warnings, re, math, numpy, pandas, matplotlib, seaborn, cv2, pydicom, tqdm, sklearn, torch, torchvision

In [1]:
import os
import gc
import random
import warnings
import re
import math

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import pydicom

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from torchvision.models import (
    resnet50,
    ResNet50_Weights
)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128


### Set device to GPU if available else CPU, check GPU properties

In [2]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory / 1024**3,
            2
        ),
        "GB"
    )

Device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


### Set directory names

In [3]:
INPUT_ROOT = "/kaggle/input"

print(
    os.listdir(INPUT_ROOT)
)

DATA_DIR = (
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

print(
    os.listdir(DATA_DIR)
)

['competitions', 'datasets']
['test_series.csv', 'train_series.csv', 'sample_submission.csv', 'test_series', 'train_series', 'train.csv', 'test.csv']


### Read data files

In [4]:
# train = pd.read_csv(
#     os.path.join(
#         DATA_DIR,
#         "train.csv"
#     )
# )

# train_series = pd.read_csv(
#     os.path.join(
#         DATA_DIR,
#         "train_series.csv"
#     )
# )

test = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "test.csv"
    )
)

test_series = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "test_series.csv"
    )
)

sample_submission = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "sample_submission.csv"
    )
)

# print(
#     "Train:",
#     train.shape
# )

# print(
#     "Train series:",
#     train_series.shape
# )

# print(
#     "Test:",
#     test.shape
# )

# print(
#     "Test series:",
#     test_series.shape
# )

# print(
#     "Sample submission:",
#     sample_submission.shape
# )

### Save targets, print stats

In [5]:
TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

NUM_CLASSES = len(TARGETS)

print(
    "Number of targets:",
    NUM_CLASSES
)

for i, target in enumerate(TARGETS):
    print(i, target)

# print(
#     "Total studies:",
#     len(train)
# )

# print(
#     "\nNon-null labels:"
# )

# print(
#     train[TARGETS].notna().sum()
# )

Number of targets: 12
0 ACL
1 MCL
2 Medial Meniscus
3 Lateral Meniscus
4 Medial OA
5 Lateral OA
6 PF OA
7 Effusion
8 Synovitis
9 Baker's
10 Contusion
11 Fracture


### Save data file paths

In [6]:
TRAIN_SERIES_DIR = os.path.join(
    DATA_DIR,
    "train_series"
)

TEST_SERIES_DIR = os.path.join(
    DATA_DIR,
    "test_series"
)

print(
    TRAIN_SERIES_DIR
)

print(
    TEST_SERIES_DIR
)

/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series
/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series


### Define function to get dicom paths

In [7]:
def get_dicom_paths(
    study_id,
    series_id,
    base_dir
):

    series_path = os.path.join(
        base_dir,
        str(study_id),
        str(series_id)
    )

    if not os.path.exists(
        series_path
    ):
        return []

    paths = []

    for filename in os.listdir(
        series_path
    ):

        if filename.lower().endswith(
            ".dcm"
        ):

            paths.append(
                os.path.join(
                    series_path,
                    filename
                )
            )

    return paths

### Define function to get sorted dicom paths based on instance number of dcm

In [8]:
def get_sorted_dicom_paths(
    study_id,
    series_id,
    base_dir
):

    paths = get_dicom_paths(
        study_id,
        series_id,
        base_dir
    )

    records = []

    for path in paths:

        try:

            dcm = pydicom.dcmread(
                path,
                stop_before_pixels=True
            )

            instance_number = getattr(
                dcm,
                "InstanceNumber",
                0
            )

            records.append(
                (
                    instance_number,
                    path
                )
            )

        except Exception:
            continue

    records.sort(
        key=lambda x: x[0]
    )

    return [
        path
        for _, path in records
    ]

### Define function to read dicom image as array, resize, and normalize

In [9]:
def read_dicom(
    path
):

    dcm = pydicom.dcmread(
        path
    )

    image = dcm.pixel_array.astype(
        np.float32
    )

    if getattr(
        dcm,
        "PhotometricInterpretation",
        ""
    ) == "MONOCHROME1":

        image = (
            image.max() - image
        )

    # Apply rescale slope/intercept
    slope = float(
        getattr(
            dcm,
            "RescaleSlope",
            1
        )
    )

    intercept = float(
        getattr(
            dcm,
            "RescaleIntercept",
            0
        )
    )

    image = (
        image * slope
        + intercept
    )

    # Percentile normalization
    low = np.percentile(
        image,
        1
    )

    high = np.percentile(
        image,
        99
    )

    if high > low:

        image = (
            image - low
        ) / (
            high - low
        )

    else:

        image = np.zeros_like(
            image
        )

    image = np.clip(
        image,
        0,
        1
    )

    return image

### Define function to return max num_slices evenly spaced image paths

In [10]:
def select_representative_slices(
    paths,
    num_slices=8
):

    if len(paths) == 0:
        return []

    if len(paths) <= num_slices:

        return paths

    indices = np.linspace(
        0,
        len(paths) - 1,
        num_slices
    ).astype(int)

    return [
        paths[i]
        for i in indices
    ]

### Define function to return max_series series from one study based on priority order

In [11]:
def select_series(
    study_id,
    series_df,
    max_series=3
):

    df = series_df[
        series_df[
            "StudyInstanceUID"
        ] == study_id
    ].copy()

    if len(df) == 0:
        return df

    # Priority:
    # 1. Fluid-sensitive
    # 2. Fat suppression
    # 3. Anatomical plane

    df["priority"] = (
        df["Fluid_Sensitive"].astype(int)
        * 2
        +
        df["Fat_Suppression"].astype(int)
    )

    df = df.sort_values(
        "priority",
        ascending=False
    )

    return df.head(
        max_series
    )

### Define image transforms
Resize, horizontal flip, rotate, normalize for training data \
Resize, normalize for validation data

In [12]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        8
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

valid_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [13]:
import h5py

def build_hdf5_cache(df, series_df, base_dir, h5_path, num_slices=8, max_series=2, image_size=IMAGE_SIZE):
    study_ids = df["StudyInstanceUID"].unique()

    with h5py.File(h5_path, "w") as h5f:
        images_ds = h5f.create_dataset(
            "images",
            shape=(0, image_size, image_size),
            maxshape=(None, image_size, image_size),
            chunks=(1, image_size, image_size),
            dtype=np.uint8,
        )
        study_ds = h5f.create_dataset(
            "study_ids",
            shape=(0,),
            maxshape=(None,),
            dtype=h5py.string_dtype(),
        )

        row = 0

        for study_id in tqdm(study_ids, desc=f"Caching {os.path.basename(h5_path)}"):
            series_info = select_series(study_id, series_df, max_series)

            for _, series_row in series_info.iterrows():
                series_id = series_row["SeriesInstanceUID"]
                paths = get_sorted_dicom_paths(study_id, series_id, base_dir)
                selected_paths = select_representative_slices(paths, num_slices)

                for path in selected_paths:
                    try:
                        image = read_dicom(path)
                    except Exception:
                        continue

                    image = cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_AREA)
                    image = (image * 255).clip(0, 255).astype(np.uint8)

                    images_ds.resize(row + 1, axis=0)
                    images_ds[row] = image

                    study_ds.resize(row + 1, axis=0)
                    study_ds[row] = str(study_id)

                    row += 1

        print(f"Wrote {row} slices to {h5_path}")

### Define dataset class
Returns dataset object

In [14]:
class HDF5KneeDataset(Dataset):
    def __init__(self, dataframe, h5_path, targets=None, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.h5_path = h5_path
        self.targets = targets
        self.transform = transform
        self._h5 = None  # opened lazily per worker, not here

        with h5py.File(self.h5_path, "r") as h5f:
            study_ids = h5f["study_ids"][:].astype(str)

        self.study_to_rows = {}
        for row_idx, study_id in enumerate(study_ids):
            self.study_to_rows.setdefault(study_id, []).append(row_idx)

    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = str(row["StudyInstanceUID"])
        rows = self.study_to_rows.get(study_id, [])
        h5f = self._get_h5()
    
        if len(rows) == 0:
            images = torch.zeros(
                16, 3, IMAGE_SIZE, IMAGE_SIZE
            )
    
        else:
            # Select 8 equally spaced slices
            indices = np.linspace(
                0,
                len(rows) - 1,
                16
            ).round().astype(int)
    
            selected_rows = [rows[i] for i in indices]
    
            image_list = []
    
            for selected_row in selected_rows:
                array = h5f["images"][selected_row]
    
                # Grayscale -> 3 channels for ImageNet-pretrained ResNet
                array = np.stack(
                    [array, array, array],
                    axis=-1
                )
    
                image = (
                    self.transform(array)
                    if self.transform
                    else torch.from_numpy(array)
                )
    
                image_list.append(image)
    
            images = torch.stack(image_list, dim=0)
    
        if self.targets is None:
            return images, study_id
    
        labels = torch.tensor(
            row[self.targets].values.astype(np.float32),
            dtype=torch.float32
        )
    
        return images, labels

### Split official data into training and validation

In [15]:
CACHE_DIR = "/kaggle/working/hdf5_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# TRAIN_H5 = os.path.join(CACHE_DIR, "train_cache.h5")
# VALID_H5 = os.path.join(CACHE_DIR, "valid_cache.h5")
TEST_H5 = os.path.join(CACHE_DIR, "test_cache.h5")

# build_hdf5_cache(pseudo_training, train_series, TRAIN_SERIES_DIR, TRAIN_H5, num_slices=8, max_series=2)
# build_hdf5_cache(labeled_data, train_series, TRAIN_SERIES_DIR, VALID_H5, num_slices=8, max_series=2)
build_hdf5_cache(test, test_series, TEST_SERIES_DIR, TEST_H5, num_slices=8, max_series=2)

# from IPython.display import FileLink

# FileLink('/kaggle/working/hdf5_cache/train_cache.h5')
# FileLink('/kaggle/working/hdf5_cache/valid_cache.h5')
# FileLink('/kaggle/working/hdf5_cache/test_cache.h5')

Caching test_cache.h5:   0%|          | 0/3 [00:00<?, ?it/s]

Wrote 48 slices to /kaggle/working/hdf5_cache/test_cache.h5


### Define model class

In [16]:
class KneeResNet(nn.Module):
    def __init__(
        self,
        num_classes=12,
        num_series=2,
        slices_per_series=8,
        transformer_dim=512,
        num_heads=8,
        num_layers=2
    ):
        super().__init__()

        self.num_series = num_series
        self.slices_per_series = slices_per_series
        self.num_slices = num_series * slices_per_series

        # -------------------------
        # ResNet-50 feature extractor
        # -------------------------
        self.backbone = resnet50(
            weights=None
        )

        features_dim = self.backbone.fc.in_features  # 2048

        self.backbone.fc = nn.Identity()

        # -------------------------
        # Project ResNet features
        # 2048 -> 512
        # -------------------------
        self.feature_projection = nn.Linear(
            features_dim,
            transformer_dim
        )

        # -------------------------
        # CLS token
        # -------------------------
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, transformer_dim)
        )

        # -------------------------
        # Positional embeddings
        # -------------------------
        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.num_slices + 1,
                transformer_dim
            )
        )

        # -------------------------
        # Transformer
        # -------------------------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # -------------------------
        # Classifier
        # -------------------------
        self.classifier = nn.Linear(
            transformer_dim,
            num_classes
        )

    def forward(self, x):
        # x: [B, 16, C, H, W]

        B, N, C, H, W = x.shape

        # Convert:
        # [B, 16, C, H, W]
        # ->
        # [B, 2, 8, C, H, W]
        x = x.view(
            B,
            self.num_series,
            self.slices_per_series,
            C,
            H,
            W
        )

        # Flatten the series/slice dimensions
        # for ResNet
        x = x.view(
            B * self.num_slices,
            C,
            H,
            W
        )

        # ResNet feature extraction
        features = self.backbone(x)
        # [B*16, 2048]

        # Project to Transformer dimension
        features = self.feature_projection(features)
        # [B*16, 512]

        # Restore sequence structure
        features = features.view(
            B,
            self.num_slices,
            -1
        )
        # [B, 16, 512]

        # -------------------------
        # Add CLS token
        # -------------------------
        cls_token = self.cls_token.expand(B, -1, -1)
        # [B, 1, 512]

        features = torch.cat(
            [cls_token, features],
            dim=1
        )
        # [B, 17, 512]

        # Add positional embeddings
        features = features + self.pos_embedding

        # Transformer
        features = self.transformer(features)
        # [B, 17, 512]

        # Take CLS representation
        features = features[:, 0]
        # [B, 512]

        # Classification
        return self.classifier(features)

### Create model object

In [17]:
model = KneeResNet(
    num_classes=NUM_CLASSES
)

model = model.to(
    DEVICE
)

print(
    "Model ready."
)

Model ready.


### Load best model in eval mode

In [18]:
model.load_state_dict(
    torch.load(
        "/kaggle/input/datasets/shantanuedgaonkar/best-knee-model/best_knee_model.pth",
        map_location=DEVICE
    )
)

model.eval()

print(
    "Best model loaded."
)

Best model loaded.


### Define test dataset

In [19]:
test_dataset = HDF5KneeDataset(
    dataframe=test,
    h5_path=TEST_H5,
    targets=None,
    transform=valid_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Test studies:", len(test_dataset))

Test studies: 3


### Generate predictions for submission

In [20]:
model.eval()

test_predictions = []

test_ids = []

with torch.no_grad():

    for images, study_ids in tqdm(
        test_loader,
        desc="Predicting"
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            images
        )

        probabilities = (
            torch.sigmoid(
                logits
            )
        )

        test_predictions.append(
            probabilities
            .cpu()
            .numpy()
        )

        test_ids.extend(
            list(study_ids)
        )

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
test_predictions = np.concatenate(
    test_predictions,
    axis=0
)

submission = pd.DataFrame(
    test_predictions,
    columns=TARGETS
)

submission.insert(
    0,
    "StudyInstanceUID",
    test_ids
)

display(
    submission.head()
)

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.196436,0.277716,0.716388,0.274321,0.472944,0.233627,0.509549,0.701230,0.190873,0.691976,0.023305,0.002022
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.136435,0.093634,0.175858,0.152980,0.289029,0.223918,0.332949,0.527996,0.222124,0.172849,0.074981,0.033992
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.091774,0.158742,0.230976,0.302236,0.313405,0.282071,0.661965,0.617915,0.108531,0.338260,0.039134,0.014706


In [22]:
print(
    "Minimum prediction:",
    submission[TARGETS]
    .min()
    .min()
)

print(
    "Maximum prediction:",
    submission[TARGETS]
    .max()
    .max()
)

Minimum prediction: 0.00202248920686543
Maximum prediction: 0.7163878679275513


In [23]:
print(
    "Sample columns:"
)

print(
    sample_submission.columns.tolist()
)

print(
    "\nOur columns:"
)

print(
    submission.columns.tolist()
)

Sample columns:
['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

Our columns:
['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


In [24]:
submission = submission[
    sample_submission.columns
]

In [25]:
SUBMISSION_PATH = (
    "/kaggle/working/"
    "submission.csv"
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

print(
    "Submission saved:"
)

print(
    SUBMISSION_PATH
)

Submission saved:
/kaggle/working/submission.csv


In [26]:
submission_check = pd.read_csv(
    SUBMISSION_PATH
)

assert list(
    submission_check.columns
) == list(
    sample_submission.columns
)

assert len(
    submission_check
) == len(
    sample_submission
)

assert (
    submission_check[
        TARGETS
    ].isna()
    .sum()
    .sum()
    == 0
)

assert (
    submission_check[
        TARGETS
    ].values >= 0
).all()

assert (
    submission_check[
        TARGETS
    ].values <= 1
).all()

print(
    "================================"
)

print(
    "SUBMISSION VALIDATION PASSED"
)

print(
    "================================"
)

display(
    submission_check.head()
)

SUBMISSION VALIDATION PASSED


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.196436,0.277716,0.716388,0.274321,0.472944,0.233627,0.509549,0.701230,0.190873,0.691976,0.023305,0.002022
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.136435,0.093634,0.175858,0.152980,0.289029,0.223918,0.332949,0.527996,0.222124,0.172849,0.074981,0.033992
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.091774,0.158742,0.230976,0.302236,0.313405,0.282071,0.661965,0.617915,0.108531,0.338260,0.039134,0.014706


In [27]:
# FINAL SUBMISSION

submission = submission[
    sample_submission.columns
].copy()

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("submission.csv created successfully!")
print(submission.shape)
display(submission.head())

submission.csv created successfully!
(3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.196436,0.277716,0.716388,0.274321,0.472944,0.233627,0.509549,0.701230,0.190873,0.691976,0.023305,0.002022
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.136435,0.093634,0.175858,0.152980,0.289029,0.223918,0.332949,0.527996,0.222124,0.172849,0.074981,0.033992
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.091774,0.158742,0.230976,0.302236,0.313405,0.282071,0.661965,0.617915,0.108531,0.338260,0.039134,0.014706


In [28]:
import os

path = "/kaggle/working/submission.csv"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("File size:", os.path.getsize(path), "bytes")
    print("Ready for Kaggle submission!")

Exists: True
File size: 719 bytes
Ready for Kaggle submission!
